## Frengression versus Frugal Flows on the paper's causl binary ATE experiment

This adds Frugal Flows to the **existing binary ATE experiment** (`xwshen51/frengression/paper_exp/binary.ipynb`). The bundled `causl_binary.R` contains its `data.causl` generator and uses the same parameters: 5 instrumental variables, 5 confounders, no outcome-only or spurious covariates, `n=5000`, and a specified ATE of **2**. Both models see the **same observational dataset** per seed. Frengression uses the paper's `num_layer=3`, `hidden_dim=100`, `noise_dim=1`, learning rate `1e-4`, and 1000 updates. The original experiment varies instrument strength over `0, 0.5, 1, 1.5, 2` and repeats 30 times.

The causal margins specified to `causl` are $Y(0)\sim N(0,1)$ and $Y(1)\sim N(2,1)$. This notebook uses their exact ATE, means, and Gaussian laws as ground truth: **it does not draw a separate Monte Carlo reference dataset**. The two fitted models are implicit/sampling models, so their reported ATEs are estimated from finite model draws (`mc=10000` by default); the output includes the associated model-draw standard error. Scalar Wasserstein and energy distances use analytic Gaussian expectations. These are exploratory results until repeated over seeds and instrument strengths.

Before running in VS Code, install R and the R package `causl` (in R: `install.packages('remotes'); remotes::install_github('rje42/causl')`). Select the Python kernel for this checkout. The first code cell installs missing Python packages into that kernel and checks the R dependency. Restart the kernel if Python dependencies change. Running all cells now runs a five-seed diagnostic grid across the original instrument strengths. Each completed seed is saved separately, so an interrupted run can resume. The one-seed pilot and 30-repetition final grid are off by default.


In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys

# Works when VS Code uses either the repository root or examples/ as cwd.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file()
             and (p / "examples" / "compare_frugal_flows_causl.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the frengression_pkg checkout in VS Code and use its notebook kernel.")

# Pinned to a Frugal Flows revision with sample_do and flexible_continuous.
FRUGAL_FLOWS_COMMIT = "26bc25cfa0c262e8a34af61841b202de79ffae0c"
FLOWJAX_VERSION = "19.1.0"  # From frugal-flows/environment.yaml.

def install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *map(str, args)])

# This repository has a src/ layout. Put the checked-out code on the live
# notebook kernel's path even if pip's editable-install metadata is stale.
sys.path.insert(0, str(ROOT / "src"))
if any(importlib.util.find_spec(name) is None for name in ("torch", "engression")):
    install("-e", ROOT)
from frengression import Frengression  # Fail here rather than during the fit.

# Frugal Flows calls fit_to_data(data=...), which older FlowJAX versions do
# not accept. Install the version in Frugal Flows' own environment file.
try:
    installed_flowjax = metadata.version("flowjax")
except metadata.PackageNotFoundError:
    installed_flowjax = None
if installed_flowjax != FLOWJAX_VERSION:
    print(f"FlowJAX {installed_flowjax!r} -> {FLOWJAX_VERSION}; installing compatible version")
    install("flowjax==" + FLOWJAX_VERSION, "lineax==0.1.0", "paramax>=0.0.5")
    importlib.invalidate_caches()

if importlib.util.find_spec("frugal_flows") is None:
    install("git+https://github.com/llaurabatt/frugal-flows.git@" + FRUGAL_FLOWS_COMMIT)
missing = [name for name in ("scipy", "pandas", "matplotlib")
           if importlib.util.find_spec(name) is None]
if missing:
    install(*missing)
from frugal_flows.benchmarking import FrugalFlowModel  # Check the API up front.
from flowjax.train import fit_to_data
import inspect
if "data" not in inspect.signature(fit_to_data).parameters:
    raise RuntimeError("Old FlowJAX remains loaded. Restart the notebook kernel and rerun from the first cell.")

sys.path.insert(0, str(ROOT / "examples"))
from compare_frugal_flows_causl import run, generate_causl, score
import numpy as np
import pandas as pd
from IPython.display import display
import shutil
if shutil.which("Rscript") is None:
    raise RuntimeError("Rscript is missing. Install R and causl; see the first notebook cell.")
check_r = subprocess.run(["Rscript", "-e", "if (!requireNamespace('causl', quietly=TRUE)) quit(status=2)"], capture_output=True, text=True)
if check_r.returncode:
    raise RuntimeError("R package causl is missing. In R, run remotes::install_github('rje42/causl').")
print("Using repository:", ROOT)
print("Python kernel:", sys.executable)


### Confirm the original experiment settings

The following small draw only verifies the R to Python connection and column order. It is **not** used to estimate the causal truth. Observed treatment-arm means are confounded and need not equal 0 and 2.


In [ ]:
x_preview, z_preview, y_preview = generate_causl(n=200, seed=2026, strength_instr=0.0)
print('Covariates:', z_preview.shape[1], '| treatment values:', np.unique(x_preview[:, 0]))
print('Observed arm counts:', {arm: int((x_preview[:, 0] == arm).sum()) for arm in (0, 1)})
print('Exact causal means: do(0)=0, do(1)=2; exact ATE=2')


### Experiment settings

You have already checked the one-seed pilot. The notebook now defaults to **five instrument strengths × five seeds** on the same original causl DGP and Frengression settings. Set `RUN_GRID_PILOT = False` to skip this staged grid. Each completed `(strength, seed)` is saved as a separate CSV; rerunning reuses only complete matching files. `RUN_PILOT = True` reruns the original one-seed pilot; `RUN_FR_DIAGNOSTIC = True` adds a 2,000-update Frengression fit after that pilot. `RUN_FULL = True` runs 30 repetitions per strength for the final paper table and is off by default. Fit seconds are diagnostic, because the two methods fit different sets of components.


In [ ]:
from argparse import Namespace

RUN_PILOT = False          # One-seed pilot already checked.
RUN_GRID_PILOT = True      # Five strengths × five seeds; saved per seed.
GRID_REPEATS = 5
RUN_FR_DIAGNOSTIC = False  # Enable only if the pilot shows underfitting.
RUN_FULL = False            # Five instrument strengths × 30 seeds × two methods.
OUTPUT_DIR = ROOT / 'examples' / 'benchmark_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

pilot = Namespace(n=5000, repeats=1, seed=2026, strength_instr=0.0,
                  fr_iters=1000, flow_epochs=2000, marginal_epochs=400,
                  mc=10000, methods=['frengression', 'frugal_flows'],
                  output=str(OUTPUT_DIR / 'causl_pilot.csv'))
print('Pilot:', vars(pilot))
print('Frengression: layers=3, hidden_dim=100, noise_dim=1, lr=1e-4')
print('Run pilot:', RUN_PILOT, '| Run five-seed grid:', RUN_GRID_PILOT,
      '| Fr diagnostic:', RUN_FR_DIAGNOSTIC, '| Full 30 × 5 grid:', RUN_FULL)


### One-seed pilot

Check both fitted arm means and signed ATE against their **exact** values. A pilot cannot establish a winner. The ATE standard error in the table reflects finite draws from the fitted model, not uncertainty from training data or across seeds.


In [ ]:
if RUN_PILOT:
    pilot_signature = (pilot.n, pilot.seed, pilot.strength_instr,
                       pilot.fr_iters, pilot.flow_epochs, pilot.marginal_epochs, pilot.mc)
    run(pilot)
    pilot_results = pd.read_csv(pilot.output)
    display(pilot_results[['method', 'n', 'strength_instr', 'true_ate', 'estimated_ate',
                           'ate_error', 'model_draw_ate_se', 'true_mean0', 'estimated_mean0',
                           'true_mean1', 'estimated_mean1', 'wasserstein_mean',
                           'energy_mean', 'quantile_mae', 'seconds']])
else:
    print('Pilot skipped. Set RUN_PILOT=True in the configuration cell.')


### Optional Frengression convergence check

If the pilot underfits, set `RUN_FR_DIAGNOSTIC=True`, rerun the configuration cell, then run this cell. It uses the **same causl data and seed** with 2000 Frengression updates and the unchanged `noise_dim=1` architecture. Inspect arm means and distributional distances; do not tune the model to one favourable seed and then report that seed as final.


In [ ]:
if RUN_FR_DIAGNOSTIC:
    current_signature = (pilot.n, pilot.seed, pilot.strength_instr,
                         pilot.fr_iters, pilot.flow_epochs, pilot.marginal_epochs, pilot.mc)
    if 'pilot_signature' not in globals() or pilot_signature != current_signature:
        raise RuntimeError('Run the pilot again after changing its configuration.')
    settings = Namespace(**vars(pilot))
    settings.fr_iters = 2000
    settings.methods = ['frengression']
    settings.output = str(OUTPUT_DIR / 'causl_fr_2000.csv')
    run(settings)
    diagnostic = pd.concat([pilot_results.loc[pilot_results.method == 'frengression'],
                            pd.read_csv(settings.output)], ignore_index=True)
    display(diagnostic[['fr_iters', 'estimated_ate', 'ate_error', 'model_draw_ate_se',
                        'estimated_mean0', 'estimated_mean1', 'wasserstein_mean',
                        'energy_mean', 'seconds']])
else:
    print('Frengression diagnostic skipped.')


### Five-seed grid: the next run

With the default `RUN_GRID_PILOT=True`, run the next cell. It uses the original instrument-strength grid `0, 0.5, 1, 1.5, 2` and five paired seeds per level. It will fit 25 Frengression models and 25 Frugal Flows models and may take **over half an hour** on a laptop. Each completed pair is saved as its own CSV, and a combined CSV is refreshed after each pair. If interrupted, rerun the cell; complete matching files are reused. The table reports signed ATE bias, MAE, RMSE, and distributional errors by strength. Five seeds show trends but are not the paper's 30-repetition result.


In [ ]:
if RUN_GRID_PILOT:
    tag = (f'n{pilot.n}_fr{pilot.fr_iters}_ff{pilot.flow_epochs}_'
           f'm{pilot.marginal_epochs}_mc{pilot.mc}')
    combined_path = OUTPUT_DIR / f'causl_grid5_{tag}.csv'
    collected = []
    for strength in np.arange(0, 2.5, 0.5):
        for rep in range(GRID_REPEATS):
            settings = Namespace(**vars(pilot))
            settings.seed = pilot.seed + rep
            settings.repeats = 1
            settings.strength_instr = float(strength)
            settings.output = str(
                OUTPUT_DIR / f'causl_grid5_{tag}_s{strength:.1f}_seed{settings.seed}.csv')
            result_file = Path(settings.output)
            if result_file.is_file():
                rows = pd.read_csv(result_file)
                if (len(rows) != 2
                    or set(rows.method) != {'frengression', 'frugal_flows'}
                    or not (rows.seed == settings.seed).all()
                    or not (rows.n == settings.n).all()
                    or not np.allclose(rows.strength_instr, strength)
                    or not (rows.true_ate == 2.0).all()):
                    raise RuntimeError(f'Unexpected cached result: {result_file}')
                print('Using completed pair:', result_file.name)
            else:
                print(f'Running strength={strength:.1f}, seed={settings.seed}')
                run(settings)
                rows = pd.read_csv(result_file)
            collected.append(rows)
            pd.concat(collected, ignore_index=True).to_csv(combined_path, index=False)
    grid_results = pd.concat(collected, ignore_index=True)
    grid_results['signed_error'] = grid_results.estimated_ate - grid_results.true_ate
    grid_summary = grid_results.groupby(['strength_instr', 'method']).agg(
        seeds=('seed', 'nunique'),
        ate_bias=('signed_error', 'mean'),
        ate_mae=('ate_error', 'mean'),
        ate_rmse=('signed_error', lambda v: float(np.sqrt(np.mean(np.square(v))))),
        wasserstein=('wasserstein_mean', 'mean'),
        energy=('energy_mean', 'mean'),
        quantile_mae=('quantile_mae', 'mean')
    )
    display(grid_summary.round(4))
    print('Combined results:', combined_path)
else:
    print('Five-seed grid skipped. Set RUN_GRID_PILOT=True and rerun the configuration cell.')


### Original 30-repetition grid for the final table

After reviewing the five-seed trends, enable `RUN_FULL=True` to run 30 repetitions for each strength. This is a separate, more expensive experiment. The full-grid files have different names and will not overwrite the five-seed checkpoints.


In [ ]:
if RUN_FULL:
    full_frames = []
    for strength in np.arange(0, 2.5, 0.5):
        settings = Namespace(**vars(pilot))
        settings.repeats = 30
        settings.strength_instr = float(strength)
        settings.output = str(OUTPUT_DIR / f'causl_strength_{strength:.1f}.csv')
        print('Running instrument strength', strength)
        run(settings)
        full_frames.append(pd.read_csv(settings.output))
    full_results = pd.concat(full_frames, ignore_index=True)
    full_results.to_csv(OUTPUT_DIR / 'causl_ff_full.csv', index=False)
    display(full_results.groupby(['strength_instr', 'method'])[
        ['ate_error', 'wasserstein_mean', 'energy_mean', 'quantile_mae']
    ].agg(['mean', 'std']).round(4))
else:
    print('Full 30 × 5 grid skipped. Set RUN_FULL=True and rerun the configuration cell.')


### Interpretation and reproducibility

Report all methods and strengths, including settings where Frugal Flows does better. Summarize signed ATE bias, RMSE, MAE and the distributional metrics over independent seeds. Fitted-model draw error is smaller than but distinct from across-seed training variation. This notebook addresses the shared binary, scalar-outcome setting. Joint outcomes and longitudinal data need separate experiments and appropriate baselines. The earlier three-covariate comparison in `compare_frugal_flows.py` remains exploratory and should not be used as the paper's binary ATE benchmark.


In [ ]:
for package in ('frengression', 'frugal-flows', 'torch', 'jax', 'flowjax', 'numpy', 'scipy'):
    try:
        print(f'{package}: {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}: version unavailable')
try:
    print('frengression_pkg commit:', subprocess.check_output(
        ['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip())
except (subprocess.CalledProcessError, FileNotFoundError):
    pass
print('Frugal Flows install target:', FRUGAL_FLOWS_COMMIT)
